# 📊 PosePulse — paper classification benchmarks (Colab T4)

Trains **both** models from the capstone paper on identical inputs:

| Model | Architecture | Params |
|---|---|---|
| **BiLSTM-CNN** | BiLSTM(4 hidden/direction) → Conv2D(128) → Conv2D(256) → Conv2D(64) → Conv2D(1) → Linear → 4-class | ~450 K |
| **xLSTM[7:1]** | 7 mLSTM + 1 sLSTM (hidden=256, heads=4) → mean-pool → Linear → 4-class | ~5.2 M |

Hyperparameters per paper §3.3.3:
- AdamW · lr=3e-4 (cosine to 3e-6) · betas=(0.9, 0.999) · eps=1e-8 · weight_decay=1e-4
- 50 epochs · batch 64 · grad-clip 1.0 · CE loss with label smoothing 0.1
- Best checkpoint by val accuracy

Wall-clock estimate on T4 (~10k windows · 50 epochs · both models): **30–50 min**.

## 1 · GPU & runtime check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch, sys
assert torch.cuda.is_available(), 'No CUDA — switch runtime to T4 GPU.'
print(f'torch {torch.__version__} · {torch.cuda.get_device_name(0)} · py {sys.version.split()[0]}')

## 2 · Mount Drive + auto-detect features

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
OUT_ROOT   = f'{DRIVE_ROOT}/paper_classification_T4'
os.makedirs(OUT_ROOT, exist_ok=True)

# Required on Drive:
#   * a directory of per-clip *.npz feature files (ViTPose-S or angle features)
#   * an index CSV with columns: video_stem, exercise_class, split (train/val/test)
FEATURES_DIR = f'{DRIVE_ROOT}/riccio_features'   # ← edit if your folder name differs
INDEX_CSV    = f'{DRIVE_ROOT}/riccio_index_split.csv'

assert os.path.isdir(FEATURES_DIR), f'features dir not found: {FEATURES_DIR}'
assert os.path.isfile(INDEX_CSV),  f'index CSV not found: {INDEX_CSV}'
print(f'features → {FEATURES_DIR}\nindex    → {INDEX_CSV}\nout      → {OUT_ROOT}')

## 3 · Pull project code (auto-detect folder name)

In [ ]:
GITHUB_URL = ''   # e.g. 'https://github.com/elinmelk/Finess-coach-capstone.git'  — '' to skip

import os, glob, shlex

def _find_project(parent='/content'):
    pats = glob.glob(f'{parent}/Finess-coach-capstone*') + glob.glob(f'{parent}/Fitness-coach-capstone*')
    for p in sorted(pats):
        if os.path.isdir(p) and os.path.isfile(os.path.join(p, 'train_paper_classification.py')):
            return p
    return None

PROJECT_DIR = _find_project()
if PROJECT_DIR is None:
    # try a zip on Drive whose name contains 'finess' or 'fitness'
    for fn in os.listdir(DRIVE_ROOT):
        low = fn.lower()
        if low.endswith('.zip') and ('finess' in low or 'fitness' in low):
            !cp {shlex.quote(os.path.join(DRIVE_ROOT, fn))} /content/_p.zip
            !unzip -q -o /content/_p.zip -d /content/
            break
    PROJECT_DIR = _find_project()
if PROJECT_DIR is None and GITHUB_URL:
    %cd /content
    !git clone {GITHUB_URL}
    PROJECT_DIR = _find_project()

assert PROJECT_DIR, 'No project found. Put a project zip on Drive OR set GITHUB_URL above.'
os.chdir(PROJECT_DIR)
print(f'project → {PROJECT_DIR}')

## 4 · Install deps + verify imports

In [ ]:
!pip -q install -e {shlex.quote(PROJECT_DIR)}
from fitness_coach.models.exercise_bilstm_model import ExerciseBiLSTMCNN
from fitness_coach.models.xlstm_model import xLSTMExerciseClassifier
print('imports OK · BiLSTM-CNN + xLSTM[7:1] both available')

## 5 · Preflight check

Verifies feature files + index CSV columns + class count before training starts.

In [ ]:
import csv, glob
from collections import Counter

rows = list(csv.DictReader(open(INDEX_CSV)))
splits = Counter(r.get('split','train') for r in rows)
classes = Counter(r.get('exercise_class','?') for r in rows)
n_npz = len(glob.glob(os.path.join(FEATURES_DIR, '*.npz')))

print(f"index rows  : {len(rows)}")
print(f"splits      : {dict(splits)}")
print(f"classes ({len(classes)}): {dict(classes)}")
print(f"feature .npz: {n_npz}")
assert len(classes) >= 2, 'Need at least 2 classes.'
assert n_npz > 0, f'No .npz feature files in {FEATURES_DIR}'
print('preflight OK')

## 6 · Train both models (paper hyperparams)

AdamW · lr 3e-4 → 3e-6 cosine · betas (0.9, 0.999) · eps 1e-8 · weight_decay 1e-4 · grad-clip 1.0 · label_smoothing 0.1 · 50 epochs · batch 64.

In [ ]:
import shlex, subprocess, time

FEATURE_DIM = 256   # 256 = ViTPose-S features; set to 42 if your .npz files are angle features

cmd = f'''python -u train_paper_classification.py \
  --index-csv {shlex.quote(INDEX_CSV)} \
  --features-dir {shlex.quote(FEATURES_DIR)} \
  --feature-dim {FEATURE_DIM} \
  --seq-len 30 --stride 15 --num-classes 4 \
  --bilstm-hidden 4 \
  --xlstm-hidden 256 --xlstm-num-heads 4 --xlstm-block-pattern mmmmmmms \
  --xlstm-conv-kernel-size 4 --xlstm-projection-factor 1.333 \
  --dropout 0.15 \
  --epochs 50 --batch-size 64 --lr 3e-4 --min-lr 3e-6 \
  --weight-decay 1e-4 --grad-clip 1.0 --label-smoothing 0.1 \
  --models bilstm xlstm \
  --output-dir {shlex.quote(OUT_ROOT)}'''

print(cmd, '\n')
t0 = time.time()
rc = subprocess.call(cmd, shell=True)
print(f'\n[exit={rc}] elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'Training failed — scroll up for the traceback.'

## 7 · Training curves (both models, side-by-side)

In [ ]:
import json, matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, color in [('bilstm_cnn', 'C0'), ('xlstm_7_1', 'C3')]:
    p = f'{OUT_ROOT}/{name}/history.json'
    if not os.path.isfile(p):
        continue
    h = json.load(open(p))
    ep = [r['epoch'] for r in h]
    ax[0].plot(ep, [r['train_loss'] for r in h], label=name, color=color)
    ax[1].plot(ep, [r['val_acc']    for r in h], label=name, color=color)
ax[0].set_title('train loss');     ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set_title('val accuracy');   ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUT_ROOT}/training_curves.png', dpi=120); plt.show()
print(f'saved → {OUT_ROOT}/training_curves.png')

## 8 · Final results table + bundle

In [ ]:
import json, tarfile
from pathlib import Path

summary = json.load(open(f'{OUT_ROOT}/summary.json'))
print(f"{'model':<14s}  {'val_acc':>10s}  {'test_acc':>10s}  {'test_f1':>10s}")
print('-' * 52)
for r in summary['runs']:
    print(f"{r['name']:<14s}  {r['best_val_acc']:>10.4f}  {r['test_acc']:>10.4f}  {r['test_f1']:>10.4f}")

bundle = Path(OUT_ROOT) / 'paper_classification_bundle.tar.gz'
with tarfile.open(bundle, 'w:gz') as tar:
    for p in Path(OUT_ROOT).rglob('*'):
        if p.is_file() and p.suffix in {'.pt', '.json', '.png'}:
            tar.add(p, arcname=p.relative_to(OUT_ROOT))
print(f'\n📦 bundle → {bundle}  ({bundle.stat().st_size/1e6:.1f} MB)')

---

Deliverables for the capstone in `MyDrive/paper_classification_T4/`:

- `bilstm_cnn/best.pt`, `bilstm_cnn/history.json`, `bilstm_cnn/metrics.json`
- `xlstm_7_1/best.pt`,  `xlstm_7_1/history.json`,  `xlstm_7_1/metrics.json`
- `summary.json` — combined results across both models
- `training_curves.png` — side-by-side training curves
- `paper_classification_bundle.tar.gz` — everything in one archive